# Prayer Time Scheduler

The purpose of this notebook is to add to Google Calendar the iqamah time at the local masjid (for me: ISBCC).

## Constants

In [2]:
# website
# https://timing.athanplus.com/masjid/widgets/monthly?masjid_id=zVKp9PLP

MASJID_ID: str = "zVKp9PLP"
GMAIL_ADDRESS: str = "awab.masroor@gmail.com"
URL: str = f"https://timing.athanplus.com/masjid/widgets/monthly?masjid_id={MASJID_ID}"
MONTH: str = "APRIL"
MASJID: str = "ISBCC"

In [3]:
PRAYER_TIMES: set = {
    "FAJR",
    "DHUHR",
    "ASR",
    "MAGHRIB",
    "ISHA",
}

### Install Dependencies

In [4]:
%pip install requests beautifulsoup4 pandas datetime
%pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ------------------------ --------------- 8.1/13.2 MB 45.7 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 45.8 MB/s eta 0:00:00
  Attempting uninstall: google-api-python-client
    Found existing installation: google-api-python-client 2.162.0
    Uninstalling google-api-python-client-2.162.0:
      Successfully uninstalled google-api-python-client-2.162.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\abmas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Code

In [5]:
# get website
import requests
response = requests.get(URL)

if response.status_code == 200:
    print("Success")
else: 
    print("Failure")

Success


In [6]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# get the table
table = soup.find_all(id="time-table3")[0]
table

<table id="time-table3">
<!--table class="full-table-sec full-table-sec2"-->
<tr>
<td class="subHeader">
                                            APRIL
                                    </td>
<td class="subHeader">
                                            SHAWWAL 
                                    </td>
<td class="subHeader">
                                            FAJR
                                    </td>
<td class="subHeader">
                                            SUNRISE
                                    </td>
<td class="subHeader">
                                            DHUHR
                                    </td>
<td class="subHeader">
                                            ASR
                                    </td>
<td class="subHeader">
                                            MAGHRIB
                                    </td>
<td class="subHeader">
                                            ISHA
                                    <

In [7]:
# get the headers (class = subHeader)
headers = table.find_all(class_="subHeader")
headers = [header.text.strip() for header in headers]
headers

['APRIL', 'SHAWWAL', 'FAJR', 'SUNRISE', 'DHUHR', 'ASR', 'MAGHRIB', 'ISHA']

In [8]:
# get the rows
rows = []
for tr in table.find_all('tr')[1:]:  # Skip the header row
    cols = tr.find_all('td')
    row = [col.get_text().strip() for col in cols]
    rows.append(row)

rows[0]

['1, TUE',
 '3',
 '5:06  |  \r\n                                                                                    5:30',
 '6:26',
 '12:48  | \r\n                                                                                    1:00',
 '4:24  | \r\n                                                                                    4:35',
 '7:12  | 7:18',
 '8:31  |  \r\n                                                                                    8:45']

In [9]:
import pandas as pd
df = pd.DataFrame(rows, columns=headers)

In [10]:
# get the dates
from datetime import datetime
dates = []
for index, row in df.iterrows():    
    # get date
    month = MONTH
    day = row[MONTH]
    year = datetime.now().year
    # parse the number of the day
    day = int(day.split()[0].replace(',', ''))
    date = datetime.strptime(f"{year} {month} {day}", "%Y %B %d")
    
    dates.append(date)    

In [11]:
new_df = pd.DataFrame(dates, columns=["Date"])

In [12]:
for prayer in PRAYER_TIMES:
    new_df[f"{prayer} Adhan"] = df[prayer].apply(lambda x: x.split()[0])
    new_df[f"{prayer} Iqamah"] = df[prayer].apply(lambda x: x.split()[2])

# remove the old columns
new_df.head(10)

,Date,MAGHRIB Adhan,MAGHRIB Iqamah,ASR Adhan,ASR Iqamah,DHUHR Adhan,DHUHR Iqamah,ISHA Adhan,ISHA Iqamah,FAJR Adhan,FAJR Iqamah
0,2025-04-01,7:12,7:18,4:24,4:35,12:48,1:00,8:31,8:45,5:06,5:30
1,2025-04-02,7:13,7:19,4:25,4:35,12:48,1:00,8:32,8:45,5:04,5:30
2,2025-04-03,7:14,7:20,4:25,4:35,12:48,1:00,8:34,8:45,5:02,5:30
3,2025-04-04,7:15,7:21,4:25,4:35,12:47,1:00,8:35,8:45,5:01,5:30
4,2025-04-05,7:16,7:21,4:26,4:40,12:47,1:00,8:36,8:50,4:59,5:15
5,2025-04-06,7:17,7:22,4:26,4:40,12:47,1:00,8:38,8:50,4:57,5:15
6,2025-04-07,7:18,7:23,4:27,4:40,12:46,1:00,8:39,8:50,4:55,5:15
7,2025-04-08,7:19,7:24,4:27,4:40,12:46,1:00,8:40,8:50,4:53,5:15
8,2025-04-09,7:21,7:26,4:28,4:40,12:46,1:00,8:42,8:50,4:51,5:15
9,2025-04-10,7:22,7:27,4:28,4:40,12:46,1:00,8:43,8:50,4:49,5:15


In [13]:
from datetime import timedelta

date_df_data = []

# go through the rows and create events
for index, row in new_df.iterrows():
    new_row = {}
    new_row['Date'] = row['Date']
        
    for col in new_df.columns.drop('Date'):
        if ("DHUHR" in col or "ASR" in col or "MAGHRIB" in col or "ISHA" in col) and "11:" not in row[col]:
            dt = datetime.combine(row['Date'], datetime.strptime(f'{row[col]}', "%I:%M").time())
            dt = dt + timedelta(hours=12)
        else:
            dt = datetime.combine(row['Date'], datetime.strptime(row[col], "%H:%M").time())
        
        new_row[col] = dt

    date_df_data.append(new_row)

date_df = pd.DataFrame(date_df_data)

In [ ]:
# grab google credentials
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

creds = None
SCOPES = ['https://www.googleapis.com/auth/calendar']

if os.path.exists('token.json'):
    if (datetime.now() - datetime.fromtimestamp(os.path.getmtime('token.json'))) > timedelta(hours=24):
        print("The creds file is over 24 hours old. Deleting creds")
        os.remove('token.json')
    else:
        creds = Credentials.from_authorized_user_file('token.json')

# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            'credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

service = build('calendar', 'v3', credentials=creds)

The creds file is over 24 hours old. Deleting creds
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=380996284341-ql71lc3fj2ini29vfs0cf84ffd131gev.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50347%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=yv1fFoyN9dTyQfqdj3f806JmJe61XP&access_type=offline


In [24]:

from datetime import datetime, timedelta

def create_event(summary: str, location: str, start: datetime, end: datetime, description: str, reminders: list):
    event = {
        'summary': summary,
        'location': location,
        'description': description,
        'start': {
            'dateTime': start.strftime("%Y-%m-%dT%H:%M:%S"),
            'timeZone': 'America/New_York',
        },
        'end': {
            'dateTime': end.strftime("%Y-%m-%dT%H:%M:%S"),
            'timeZone': 'America/New_York',
        },
        'reminders': {
            'useDefault': False,
            'overrides': reminders
        },
    }

    event = service.events().insert(calendarId='60926a588d0fee35fdeaa1efde65cdcf9d6d85e2601396a18461687ea38aadf5@group.calendar.google.com', body=event).execute()
    print('Event created: %s' % (event.get('htmlLink')))

# create_event('Fajr Salah', 'https://maps.app.goo.gl
# /kC7nVr2kWGn4pNV96', datetime.now() + timedelta(days=1),
# datetime.now() + timedelta(days=1, minutes=10), 'Fajr Salah',
# [{'method': 'popup', 'minutes': 30}, {'method': 'popup', 'minutes': 15}])

In [25]:
date_df.head(3)

,Date,MAGHRIB Adhan,MAGHRIB Iqamah,ASR Adhan,ASR Iqamah,DHUHR Adhan,DHUHR Iqamah,ISHA Adhan,ISHA Iqamah,FAJR Adhan,FAJR Iqamah
0,2025-04-01,2025-04-01 19:12:00,2025-04-01 19:18:00,2025-04-01 16:24:00,2025-04-01 16:35:00,2025-04-01 12:48:00,2025-04-01 13:00:00,2025-04-01 20:31:00,2025-04-01 20:45:00,2025-04-01 05:06:00,2025-04-01 05:30:00
1,2025-04-02,2025-04-02 19:13:00,2025-04-02 19:19:00,2025-04-02 16:25:00,2025-04-02 16:35:00,2025-04-02 12:48:00,2025-04-02 13:00:00,2025-04-02 20:32:00,2025-04-02 20:45:00,2025-04-02 05:04:00,2025-04-02 05:30:00
2,2025-04-03,2025-04-03 19:14:00,2025-04-03 19:20:00,2025-04-03 16:25:00,2025-04-03 16:35:00,2025-04-03 12:48:00,2025-04-03 13:00:00,2025-04-03 20:34:00,2025-04-03 20:45:00,2025-04-03 05:02:00,2025-04-03 05:30:00


In [26]:
# go through the rows and create events
for index, row in date_df.iterrows():
    for prayer in PRAYER_TIMES:
        prayer_name = prayer.title()
        summary = f"{prayer_name} Salah"
        location = "https://maps.app.goo.gl/kC7nVr2kWGn4pNV96"
        start = row[prayer + " Iqamah"]
        end = row[prayer + " Iqamah"] + timedelta(minutes=10)
        description = f"{prayer_name} Salah at {MASJID}"
        reminders = [{'method': 'popup', 'minutes': 30}, {'method': 'popup', 'minutes': 15}]

        create_event(summary, location, start, end, description, reminders)

Event created: https://www.google.com/calendar/event?eid=cjIydjcxOWhnazJmbGVobmphZWY5dTJvNzAgNjA5MjZhNTg4ZDBmZWUzNWZkZWFhMWVmZGU2NWNkY2Y5ZDZkODVlMjYwMTM5NmExODQ2MTY4N2VhMzhhYWRmNUBn
Event created: https://www.google.com/calendar/event?eid=ZzRoYmFxcjM5MjNldmR0Y3A2NWpvcjMzdGsgNjA5MjZhNTg4ZDBmZWUzNWZkZWFhMWVmZGU2NWNkY2Y5ZDZkODVlMjYwMTM5NmExODQ2MTY4N2VhMzhhYWRmNUBn
Event created: https://www.google.com/calendar/event?eid=ZHZsMGVhMmYwYmQ5NzZ0cm0wNzM3dTNxMzggNjA5MjZhNTg4ZDBmZWUzNWZkZWFhMWVmZGU2NWNkY2Y5ZDZkODVlMjYwMTM5NmExODQ2MTY4N2VhMzhhYWRmNUBn
Event created: https://www.google.com/calendar/event?eid=cWN1bzZuZDJyYjdyMjZxdWw0cjUyazliOTAgNjA5MjZhNTg4ZDBmZWUzNWZkZWFhMWVmZGU2NWNkY2Y5ZDZkODVlMjYwMTM5NmExODQ2MTY4N2VhMzhhYWRmNUBn
Event created: https://www.google.com/calendar/event?eid=bmk1NzBpbGR2NGMwNjN1c3UzNnNjY2JmcDQgNjA5MjZhNTg4ZDBmZWUzNWZkZWFhMWVmZGU2NWNkY2Y5ZDZkODVlMjYwMTM5NmExODQ2MTY4N2VhMzhhYWRmNUBn
Event created: https://www.google.com/calendar/event?eid=cGpxbzA5NjVlM2N0NW44OWl2cGdrdWFuO